# CC3104 – Aprendizaje por Refuerzo
## Laboratorio 1 — Task 1 — Diseño formal del MDP (Entrega Parcial)

**Integrantes:**
- Sergio Orellana — 221122
- Rodrigo Mansilla — 22611
- Ricardo Chuy — 221007

---


### Contexto

Una empresa de logística urbana opera una flota de drones de reparto en una ciudad modelada como una cuadrícula de 5x5. Cada drone parte de una base central, debe entregar paquetes en puntos designados y regresar.

Dado este contexto, diseñen formalmente el MDP que modela el problema. Respondan cada pregunta en la celda de respuesta correspondiente (pueden usar texto, fórmulas en LaTeX con `$...$`, tablas, etc.).

---
### 1. Espacio de estados 𝒮

¿Qué información debe contener el estado para que la propiedad de Markov se satisfaga razonablemente? Justifiquen **cada variable que incluyen** y **cada variable que deciden omitir**. Si omiten algo que podría ser relevante, expliquen qué consecuencia tiene esa omisión sobre la validez del modelo.

**Respuesta:**

El estado debe de contener todal la información importante y necesaria para que el agente tome la mejor decisión posible. Eso quiere decir que si el dron debe llegar, regresa y tener en cuenta cuanta batería le queda, todos esos son factores que deberían de modelarse dentro del estado.

Entonces la definición formal para el estado sería:

* s = (posición, estado_misión, nivel_batería)

Donde:
* posición ∈ {(0,0), (0,1), ..., (4,4)}, son 25 valores posibles en la cuadrícula 5x5
* estado_misión ∈ {en_camino, entregado, regresando, completo}, son 4 valores
* nivel_batería ∈ {0, 1, 2, ..., 10}, lo que serían 11 niveles discretizados por fines de simplicidad

Eso quiere decir que |S| = 25 × 4 × 11 = 1,100 estados

El problema tal vez podría simplificarse pero consideramos que este es el mínimo necesario para tener en cuenta el entorno principal.

**Justificación de cada variable:**

- Posición: El drone necesita saber dónde está en la cuadrícula para poder navegar hacia su objetivo. Sin esta información no puede tomar ninguna decisión de movimiento.

- Estado de misión: El drone tiene tres fases obligatorias, salir de la base, entregar el paquete y regresar. Sin esta variable, el drone no sabría cuál es su objetivo actual. Dos drones en la misma posición pero en fases distintas deben comportarse de manera completamente diferente.

- Nivel de batería: Sin esta variable, dos drones en la misma posición y fase de misión pero con distinta batería serían indistinguibles para el modelo, pero deberían tomar decisiones completamente distintas. Un drone con batería baja debería priorizar el regreso urgente, mientras que uno con batería alta puede continuar con normalidad. Omitir esta variable viola directamente la propiedad de Markov.


**Variables omitidas y sus consecuencias**

- Clima / viento: Se omite del estado porque su efecto se captura en la función de transición como aleatoriedad. No es necesario que el drone "sepa" el viento  solo que sus movimientos tienen cierta incertidumbre.

- Hora del día: Se omite por simplicidad del modelo. Incluirla multiplicaría el espacio de estados por 24 si tenemos en cuenta cada hora del día. Introduciríamos una variable que realmente no podemos predecir completamente, no podemos decir que una hora en específico de todos los día siempre será igual o como es que influirá diractamente al agente.

- Historial de rutas: Se omite porque con posición, misión y batería ya tenemos todo lo relevante del pasado. Saber cómo llegó el drone a donde está no cambia la decisión óptima.

---
### 2. Espacio de acciones 𝒜

Definan las acciones disponibles para el drone. ¿Es discreto o continuo? ¿Hay acciones que deberían restringirse en ciertos estados? ¿Cómo modelan esa restricción dentro del MDP?

**Respuesta:**

Ya que tenemos un espacio definido en una cuadrícula los movimientos/acciones son relativamente sencillos. Las acciones naturales son:

* A = {Norte, Sur, Este, Oeste, Esperar}

El espacio es discreto ya que el drone elige exactamente una acción en cada paso, no hay puntos medios ni combinaciones que no sean discretas. Esto se debe a que estamos en un ambiente discreto de 5x5 dond eel mvomiento es celda por celda.

**Restricciones de acciones**

- Bordes de la cuadrícula: El drone no puede salirse del perímetro definido. Si intenta moverse hacia afuera, la acción no tiene efecto el drone se queda en la misma celda y no gasta batería. Esto modela de forma natural que el drone "reconoce" el límite y no desperdicia energía intentando cruzarlo.

- Misión completa: Cuando el drone regresa exitosamente a la base tras entregar el paquete, entra en un estado terminal. En este estado no hay acciones disponibles la misión terminó.

- Batería en 0: Si el drone se queda sin batería antes de completar la misión, entra en un estado terminal de falla. Al igual que el caso anterior, no hay acciones disponibles porque el drone ya no puede operar.

Para modelarlo dentro del MDP las restriccioens deberían manejarse directamente en al función de transición. En lugar de eliminar acciones del espacio de acciones según el estado, simplemente definimos que ciertas acciones en ciertos estados producen una transición al mismo estado actual, sin cambio y sin costo de batería. Esto mantiene el espacio de acciones uniforme y simplifica la implementación. Las transiciones se mpuede limitar aplicando formulas como:
- p(s' = s | s, Norte) = 1.0 -> cuando s está en fila 0 
- p(s' = s | s, Sur) = 1.0 -> cuando s está en fila 4 
- p(s' = s | s, Este) = 1.0 -> cuando s está en columna 4 
- p(s' = s | s, Oeste) = 1.0 -> cuando s está en columna 0


---
### 3. Función de recompensa 𝑟(𝑠, 𝑎, 𝑠′)

Diseñen una función de recompensa que capture el objetivo real de la empresa. Consideren al menos tres objetivos potencialmente conflictivos: **eficiencia de entrega**, **consumo de batería** y **seguridad de vuelo**. ¿Cómo ponderan esos objetivos? ¿Qué consecuencias tendría una ponderación incorrecta sobre el comportamiento del agente?

**Respuesta:**

La función de recompensa captura los tres objetivos de la empresa:

r(s, a, s') =

- +100 -> si s' es el estado de entrega exitosa del paquete
- +50 -> si s' es el estado de regreso exitoso a la base
- -1 -> por cada paso de movimiento normal
- -100 -> si el drone llega a batería = 0 sin haber completado la misión

**Justificación de la ponderación**
- Eficiencia de entrega (+100): Es el objetivo principal del negocio. Recibe la recompensa más alta porque sin entrega no hay servicio.
- Regreso a la base (+50): Es importante operacionalmente un drone que no regresa es un drone perdido. Sin embargo es secundario al objetivo de entrega, por eso su recompensa es menor.
- Costo por paso (-1): Cada movimiento representa consumo de batería y tiempo. Esta pequeña penalización constante incentiva al agente a encontrar rutas eficientes en lugar de deambular sin rumbo.
- Batería vacía (-100): Representa un fallo operacional grave el drone cae o queda inutilizable antes de completar su misión. La penalización alta desincentiva fuertemente este resultado.

**Consecuencias de una ponderación incorrecta**

Definir mal las recompensas al final lleva a comportamientos inesperados o a "trampas" por parte del agente. El agente aprende por medio de las penalizaciones y puede encontrar trucos para poder maximizar su recompensa sin necesariamente comportarse como esperamos y qu ecumpla lo que se tenía planteado.

- Penalización por paso demasiado alta: El agente preferiría no moverse antes que arriesgarse a perder puntos. El drone se quedaría paralizado en la base indefinidamente.

- Recompensa por entrega demasiado pequeña: El agente no encontraría suficiente incentivo para alejarse de la base. Podría simplemente quedarse quieto o salir y regresar inmediatamente sin entregar.

- Sin penalización por batería vacía: El agente ignoraría completamente su nivel de batería al planificar rutas. Tomaría caminos ineficientes, se quedaría sin energía en medio de la misión y nunca regresaría a la base.


---
### 4. Función de transición 𝑝(𝑠′ ∣ 𝑠, 𝑎)

¿Es determinista o estocástica? Si es estocástica, ¿qué fuentes de aleatoriedad existen en el dominio real y cómo las modelan? Escriban **al menos tres transiciones concretas** con sus probabilidades y justifiquen cada valor.

**Respuesta:**

Aunque el sistema es discreto no necesariametne es deterministra, tomando en cuenta que el dron puede encontrarse con situaciones adversas, como por ejemplo el clima, puede que en algunos casos no se haga la decisión que el agente quiere. La función de transición es estocástica. En el mundo real, el movimiento del drone no es perfectamente predecible hay factores externos e internos que pueden causar desviaciones.

**Fuentes de aleatoriedad**

- Viento: Es impredecible y cambia constantemente. Puede desviar al drone hacia los lados de su dirección objetivo.
- Fallos internos: El GPS o los motores del drone pueden tener pequeñas imprecisiones en cualquier momento, causando que el drone no responda exactamente como se le indicó.

**Modelo de transición general**

Para un movimiento normal, las probabilidades son:

- 0.8 -> el drone llega a donde intentó ir
- 0.1 -> se desvía a la izquierda de su dirección
- 0.1 -> se desvía a la derecha de su dirección


### Transiciones concretas

| s | a | s' | p(s'\|s,a) | Justificación |
|---|---|----|------------|---------------|
| (2,2), en_camino, bat=5 | Norte | (1,2), en_camino, bat=4 | 0.8 | Movimiento exitoso, gasta 1 de batería |
| (2,2), en_camino, bat=5 | Norte | (2,1), en_camino, bat=4 | 0.1 | Viento o fallo interno desvía al Oeste |
| (2,2), en_camino, bat=5 | Norte | (2,3), en_camino, bat=4 | 0.1 | Viento o fallo interno desvía al Este |
| (0,2), en_camino, bat=5 | Norte | (0,2), en_camino, bat=5 | 1.0 | Borde superior, el drone no se mueve ni gasta batería |
| (2,2), en_camino, bat=0 | cualquiera | (2,2), falla, bat=0 | 1.0 | Batería vacía, el drone entra en estado de falla |


---
### 5. Factor de descuento 𝛾

Propongan un valor y justifíquenlo en términos del problema, no solo en términos matemáticos.

**Respuesta:**

**Respuesta:**

Valor propuesto: γ = 0.95

**Justificación**

El factor de descuento de 0.95 significa que una recompensa en el siguiente paso vale el 95% de lo que valdría ahora mismo. Este valor se eligoo considerando las características específicas del problema:

- No es bueno elegir un balor muy bajo ya que por ejemplo con γ = 0.5 por ejemplo, a los 10 pasos una recompensa de +100 valdría prácticamente 0. En una cuadrícula 5x5 donde el drone necesita varios pasos para completar la misión (salir, entregar y regresar), el agente perdería motivación para planificar a largo plazo y solo buscaría recompensas inmediatas.

- Tampoco debería ser γ = 1, sin descuento el drone trataría todas las recompensas futuras exactamente igual que las inmediatas, sin ninguna urgencia. Con batería limitada eso es problemático el drone no priorizaría terminar la misión eficientemente.

